# Monitoramento e Estratificação de Risco Neonatal (Ceará)
**Projeto Integrador UFMS - Ciência de Dados**

Este notebook implementa um pipeline End-to-End para prever o risco de óbito neonatal (0 a 27 dias) utilizando dados do SINASC e SIM.

A mortalidade neonatal é um evento raro e binário, logo a escolha do modelo precisa respeitar a natureza dos dados.

A Regressão Logística foi escolhida porque:

1) Gera saídas probabilísticas via função sigmoide (0 <= P <= 1)

2) Fornece Odds Ratios (Razão de Chances), garantindo a explicabilidade clínica exigida para validação epidemiológica.

In [1]:
# Importação das bibliotecas principais
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Configuração visual do Pandas para exibir outputs mais limpos
pd.set_option('display.max_columns', None)

# Conexão com banco de dados PostgreSQL

A etapa que requer maior processamento de dados, como o linkage determinístico, já foi realizada diretamente no SGBD.

Nessa etapa, o foco será apenas a extração da base de dados limpa, facilitando a modelagem preditiva e garantindo alta performance na memória local.

In [2]:
# Conexão com a instância do PostgreSQL no ZimaOS
# Substitua as credenciais caso necessário
DB_USER = "casaos"
DB_PASS = "casaos"
DB_HOST = "192.168.4.2"
DB_PORT = "5432"
DB_NAME = "casaos"

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Extração da coorte completa
query = "SELECT peso, apgar5, semana_gestacao, idade_mae, sexo, obito_neonatal FROM analytics.base_neonatal_pura;"
df = pd.read_sql(query, con=engine)

print(f"Volume carregado: {len(df):,} registros.")
df.head()

Volume carregado: 594,740 registros.


,peso,apgar5,semana_gestacao,idade_mae,sexo,obito_neonatal
0,2564,NaN,40.0,23,F,0
1,2812,9.0,38.0,19,F,0
2,2882,10.0,39.0,18,F,0
3,2830,9.0,39.0,12,F,0
4,2804,NaN,38.0,26,M,0


# Tratamento de Valores Ausentes e Códigos de Ignorado do SUS

Nos dados administrativos do DATASUS, campos em branco ou ignorados frequentemente vêm preenchidos com códigos numéricos padrão (como 99 para o apgar5). Se esses códigos forem mantidos como números inteiros, o modelo interpretará erroneamente que o recém-nascido teve uma vitalidade fisiológica máxima de nota "99".

Nesta etapa, convertemos o valor 99 para NaN (Not a Number) e realizamos a limpeza dos registros incompletos.

In [4]:
# Retorna as linhas que contém o valor 99
linhas_com_99 = df[(df == 99).any(axis=1)]
print(linhas_com_99)

        peso  apgar5  semana_gestacao  idade_mae sexo  obito_neonatal
31243   3466    99.0             40.0         25    M               0
31268   2200    99.0             39.0         28    F               0
31363   2700    99.0             40.0         44    F               0
31398   2370    99.0             37.0         33    F               0
60420   3506    99.0             41.0         32    M               0
...      ...     ...              ...        ...  ...             ...
581348   536    99.0             23.0         27    M               0
584407  3144    99.0             38.0         20    M               0
584426  3190    99.0             40.0         28    F               0
587766  1000    99.0              NaN         20    F               1
590680  3710    99.0             31.0         22    M               0

[94 rows x 6 columns]


In [5]:
# Retorna as colunas que contém o valor 99
colunas_com_99 = df.columns[(df == 99).any()]
print(colunas_com_99)

Index(['apgar5'], dtype='str')


In [6]:
# 1. Substitui o código de ignorado do SUS (99) por NaN
df['apgar5'] = df['apgar5'].replace(99, np.nan)

# 2. Diagnóstico de valores nulos na coorte
print("--- CONTAGEM DE VALORES NULOS POR COLUNA ---")
print(df.isnull().sum())

# 3. Remoção de registros incompletos nas variáveis preditoras cruciais
df_limpo = df.dropna().copy()

print(f"\nVolume total retido após a limpeza: {len(df_limpo):,} registros.")
print(f"Total de óbitos neonatais mantidos: {df_limpo['obito_neonatal'].sum():,}")

--- CONTAGEM DE VALORES NULOS POR COLUNA ---
peso                   0
apgar5              4713
semana_gestacao    15765
idade_mae              0
sexo                   0
obito_neonatal         0
dtype: int64

Volume total retido após a limpeza: 574,690 registros.
Total de óbitos neonatais mantidos: 3,407


# Pré-Processamento, One-Hot Encoding e Divisão Treino/Teste

Tendo em vista que os algoritmos estatísticos necessitam de entradas puramente numéricas, será necessário aplicar a técnica de *One-Hot Encoding* na variável categórica sexo ('M' e 'F'), convertendo-a em uma coluna binária (sexo_M).

Na sequência divide-se os dados em conjuntos de Treino (80%) e Teste (20%), aplicando aamostragem estratificada (stratify = y) para garantir que a proporção exata de óbitos neonatais seja preservada em ambos os conjuntos.

In [7]:
from sklearn.model_selection import train_test_split

# 1. Separação entre preditoras (X) e variável alvo (y)
X_raw = df_limpo[['peso', 'apgar5', 'semana_gestacao', 'idade_mae', 'sexo']]
y = df_limpo['obito_neonatal']

# 2. One-Hot Encoding (drop_first=True evita colinearidade perfeita)
X = pd.get_dummies(X_raw, columns=['sexo'], drop_first=True)

# 3. Divisão Estratificada em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20, # 20% reservado para teste
    random_state=42, # mantém a divisão exata dos dados sempre que executado
    stratify=y # evita que a base de teste fique subrepresentada para um valor de y
)

print(f"Base de Treino: {X_train.shape[0]:,} linhas | Óbitos no Treino: {y_train.sum():,}")
print(f"Base de Teste:  {X_test.shape[0]:,} linhas | Óbitos no Teste:  {y_test.sum():,}")
X_train.head()

Base de Treino: 459,752 linhas | Óbitos no Treino: 2,726
Base de Teste:  114,938 linhas | Óbitos no Teste:  681


,peso,apgar5,semana_gestacao,idade_mae,sexo_IGN,sexo_M
121541,2920,9.0,38.0,29,False,True
69906,2955,9.0,37.0,31,False,True
119012,3190,10.0,40.0,29,False,False
516924,3935,9.0,38.0,29,False,True
463972,3065,10.0,38.0,30,False,False


# Treinamento da Regressão Logística e Métricas Clínicas

Devido ao desbalanceamento extremo (menos de 1% de óbitos na base), é usado o hiperparâmetro class_weight='balanced'. Ele ajusta automaticamente os pesos inversamente proporcionais às frequências das classes, penalizando fortemente os erros cometidos contra a classe de óbitos.

A avaliação principal foca na métrica ROC-AUC e na Sensibilidade (Recall), prioritárias na saúde pública para evitar falsos negativos.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Instanciação e Treinamento do Modelo
modelo = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

# 2. Predição no Conjunto de Teste
y_pred = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]

# 3. Exibição das Métricas
print("--- AVALIAÇÃO DE DESEMPENHO PREDITIVO ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")

print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['Sobrevivente', 'Óbito Neonatal']))

--- AVALIAÇÃO DE DESEMPENHO PREDITIVO ---
ROC-AUC Score: 0.9544

Matriz de Confusão:
[[108792   5465]
 [    90    591]]

Relatório de Classificação Detalhado:
                precision    recall  f1-score   support

  Sobrevivente       1.00      0.95      0.98    114257
Óbito Neonatal       0.10      0.87      0.18       681

      accuracy                           0.95    114938
     macro avg       0.55      0.91      0.58    114938
  weighted avg       0.99      0.95      0.97    114938



# Estratificação em Grupos de Risco (Baixo, Médio e Alto Risco)

Ao invés de classificar os bebês de forma rígida em 0 ou 1, aplica-se o modelo para calcular a probabilidade contínua de risco (0 <= P <= 1) de toda a coorte.

Em seguida segregam-se os recém-nascidos em três categorias de risco baseadas em percentis da distriuição populacional.

Como a base de dados é bastante desbalanceada haja vista que a imensa maioria dos recém-nascidos sobrevivem, as probabilidades de cada categoria levará em consideração o peso e o índice apgar da seguinte maneira:

- Alto Risco (Top 5% -> Percentil 95 a 100): peso inferior a 1000g e Apgar5 baixo
- Médio Risco (15% Intermediários -> Percentil 80 a 95%): peso entre 1500g e 2499g, Apgar5 entre 4 e 7
- Baixo Risco (80% Inferiores -> Percentil 0 a 80): Peso >= 2500g, Apgar5 >= 8

In [9]:
# 1. Cálculo da probabilidade individual de risco para a base completa
df_limpo['probabilidade_obito'] = modelo.predict_proba(X)[:, 1]

# 2. Estratificação em percentis populacionais (80% Baixo, 15% Médio, 5% Alto Risco)
df_limpo['grupo_risco'] = pd.qcut(
    df_limpo['probabilidade_obito'],
    q=[0, 0.80, 0.95, 1.0],
    labels=['Baixo Risco', 'Médio Risco', 'Alto Risco']
)

print("--- DISTRIBUIÇÃO DA POPULAÇÃO NOS GRUPOS DE RISCO ---")
print(df_limpo['grupo_risco'].value_counts())

print("\n--- TAXA REAL DE ÓBITO OBSERVADA EM CADA GRUPO ---")
print(df_limpo.groupby('grupo_risco')['obito_neonatal'].agg(['count', 'sum', 'mean']).rename(columns={'count': 'Total Bebês', 'sum': 'Óbitos', 'mean': 'Taxa Real'}))

--- DISTRIBUIÇÃO DA POPULAÇÃO NOS GRUPOS DE RISCO ---
grupo_risco
Baixo Risco    459752
Médio Risco     86203
Alto Risco      28735
Name: count, dtype: int64

--- TAXA REAL DE ÓBITO OBSERVADA EM CADA GRUPO ---
             Total Bebês  Óbitos  Taxa Real
grupo_risco                                
Baixo Risco       459752     222   0.000483
Médio Risco        86203     230   0.002668
Alto Risco         28735    2955   0.102836


# Explicabilidade Clínica via Razão de Chances (Odds Ratio)
A validação médica do modelo exige interpretar os coeficientes (β).

Aplicando a função exponencial (e^β/β),os pesos do modelo são convertidos em Odds Ratios (Razão de Chances).

Valores de Odds Ratio maiores que 1 indicam fatores de risco (aumentam a chance de óbito), enquanto valores menores que 1 representam fatores de proteção fisiológica.

In [10]:
# 1. Extração dos Coeficientes Beta
coeficientes = modelo.coef_[0]

# 2. Cálculo dos Odds Ratios (e^Beta)
odds_ratios = np.exp(coeficientes)

# 3. Consolidação em Tabela Explicativa
df_explicabilidade = pd.DataFrame({
    'Atributo Clínico': X.columns,
    'Coeficiente Beta (β)': coeficientes,
    'Odds Ratio (e^β)': odds_ratios
}).sort_values(by='Odds Ratio (e^β)', ascending=False)

print("--- IMPACTO E FATOR DE RISCO POR VARIÁVEL ---")
df_explicabilidade.reset_index(drop=True)

--- IMPACTO E FATOR DE RISCO POR VARIÁVEL ---


,Atributo Clínico,Coeficiente Beta (β),Odds Ratio (e^β)
0,sexo_IGN,2.430184,11.360973
1,sexo_M,0.221625,1.248104
2,peso,-0.001324,0.998677
3,idade_mae,-0.002413,0.997590
4,semana_gestacao,-0.097776,0.906852
5,apgar5,-1.206057,0.299375


# Modelo Pronto

In [12]:
import joblib

# 1. Calcula as probabilidades preditas na base tratada para extrair os cortes exatos
probabilidades = modelo.predict_proba(X)[:, 1]
p80_corte = float(np.percentile(probabilidades, 80))
p95_corte = float(np.percentile(probabilidades, 95))

# 2. Empacota o modelo ajustado, as colunas esperadas e os pontos de corte em um único dicionário
artefato_modelo = {
    'modelo': modelo,
    'p80': p80_corte,
    'p95': p95_corte,
    'colunas': list(X.columns)
}

# 3. Salva o artefato em disco no diretório atual
joblib.dump(artefato_modelo, 'modelo_neonatal.pkl')

print("✓ Arquivo 'modelo_neonatal.pkl' gerado com sucesso!")
print(f"  - Limiar Baixo/Médio (P80): {p80_corte:.4f}")
print(f"  - Limiar Médio/Alto  (P95): {p95_corte:.4f}")

✓ Arquivo 'modelo_neonatal.pkl' gerado com sucesso!
  - Limiar Baixo/Médio (P80): 0.1886
  - Limiar Médio/Alto  (P95): 0.5206
